# MewtwoLLM — Full Training Pipeline
## Train a 130M param LLM from scratch on free T4 GPU

**Architecture:** RoPE + RMSNorm + SwiGLU + GQA

**Pipeline:** Download Data → Tokenize → Pretrain → SFT → DPO → Eval

**Runtime:** Change to GPU: Runtime → Change runtime type → T4 GPU

## Step 0: Setup — Clone repo + install deps

In [ ]:
#@title Clone MewtwoLLM repo { display-mode: "form" }
# Free up disk space first
!rm -rf /content/sample_data
!rm -rf ~/.cache/huggingface/datasets/*

!git clone https://github.com/superduperpiyuxh/MewtwoLLM.git
%cd MewtwoLLM
!pip install torch sentencepiece huggingface_hub datasets numpy --quiet

In [ ]:
#@title Verify GPU { display-mode: "form" }
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow!')

## Step 1: Download OpenWebText from HuggingFace

In [ ]:
#@title Download OpenWebText sample { display-mode: "form" }
#@markdown Number of samples (more = better but uses more disk/RAM)
NUM_SAMPLES = 10000  #@param {type:"integer"}

from datasets import load_dataset
import os

os.makedirs('data/raw', exist_ok=True)

# Stream dataset to avoid caching entire thing
print(f'Loading {NUM_SAMPLES} OpenWebText samples (streaming)...')
ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)

output_path = 'data/raw/openwebtext_sample.txt'
count = 0
with open(output_path, 'w') as f:
    for item in ds:
        text = item['text'].strip()
        if len(text) > 100:
            f.write(text + '\n\n')
            count += 1
        if count >= NUM_SAMPLES:
            break
        if count % 100 == 0 and count > 0:
            print(f'  {count}/{NUM_SAMPLES}...')

size_mb = os.path.getsize(output_path) / 1e6
print(f'Saved: {output_path} ({size_mb:.1f} MB, {count} docs)')

## Step 2: Train SentencePiece tokenizer

In [ ]:
#@title Train tokenizer on collected data { display-mode: "form" }
import os, glob

# Combine all text for tokenizer training
all_files = glob.glob('data/raw/**/*', recursive=True)
all_files = [f for f in all_files if os.path.isfile(f) and not f.endswith(('.bin', '.npy'))]

combined = []
for fpath in sorted(all_files):
    try:
        with open(fpath, 'r', errors='ignore') as f:
            text = f.read()
            if len(text.strip()) > 100:
                combined.append(text)
    except:
        pass

with open('data/tokenizer_input.txt', 'w') as f:
    f.write('\n'.join(combined))

print(f'Tokenizer corpus: {len(combined)} chunks, {sum(len(c) for c in combined):,} chars')

# Train SentencePiece
from src.tokenizer.tokenizer import MewtwoTokenizer

os.makedirs('data/tokenizer', exist_ok=True)
tokenizer = MewtwoTokenizer(vocab_size=32000)
tokenizer.train(input_file='data/tokenizer_input.txt', model_prefix='data/tokenizer/mewtwo')

# Verify
test = 'Hello, this is a test of the MewtwoLLM tokenizer!'
tokens = tokenizer.encode(test)
print(f'\nTest: "{test}"')
print(f'Tokens: {tokens}')
print(f'Decoded: "{tokenizer.decode(tokens)}"')
print(f'Vocab size: {tokenizer.vocab_size}')

## Step 3: Pretrain MewtwoLLM on GPU

In [ ]:
#@title Pretrain (1000 steps on T4 GPU) { display-mode: "form" }
import torch
import sys, os

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import MewtwoTokenizer
from src.training.pretrain import train

config = MewtwoConfig()
config.total_steps = 1000
config.batch_size = 8
config.gradient_accumulation_steps = 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
config.device = device
print(f'Device: {device}')

model = MewtwoLLM(config)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.1f}M)')

# Find tokenized data
data_path = None
for fname in ['data/raw/openwebtext_sample.txt', 'data/tokenizer_input.txt']:
    if os.path.exists(fname):
        data_path = fname
        break

if data_path is None:
    print('ERROR: No training data found. Run Step 1 first.')
else:
    print(f'Training data: {data_path}')
    tokenizer_path = 'data/tokenizer/mewtwo.model' if os.path.exists('data/tokenizer/mewtwo.model') else None
    train(config, data_path, checkpoint_dir='checkpoints/pretrain', tokenizer_path=tokenizer_path)

## Step 4: Supervised Fine-Tuning

In [ ]:
#@title SFT on instruction dataset { display-mode: "form" }
import json, os, sys, torch

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.alignment.sft import train_sft

# Create instruction dataset
os.makedirs('data/instruction', exist_ok=True)
instructions = [
    {'instruction': 'Explain what a transformer is in ML.', 'input': '', 'output': 'A transformer is a neural network architecture using self-attention.'},
    {'instruction': 'What is DPO?', 'input': '', 'output': 'Direct Preference Optimization directly optimizes a language model using human preference data.'},
    {'instruction': 'What does RoPE do?', 'input': '', 'output': 'Rotary Position Embedding encodes position by rotating query and key vectors.'},
    {'instruction': 'Why RMSNorm over LayerNorm?', 'input': '', 'output': 'RMSNorm removes mean centering, normalizing only by root mean square.'},
    {'instruction': 'What is SwiGLU?', 'input': '', 'output': 'SwiGLU combines Swish activation with a Gated Linear Unit in the FFN.'},
]
with open('data/instruction/instructions.jsonl', 'w') as f:
    for item in instructions:
        f.write(json.dumps(item) + '\n')

config = MewtwoConfig()

# Find latest checkpoint
ckpt_path = None
for name in ['mewtwo_final.pt', 'mewtwo_best.pt', 'model.pt']:
    p = f'checkpoints/pretrain/{name}'
    if os.path.exists(p):
        ckpt_path = p
        break

if ckpt_path:
    train_sft(
        config=config,
        model_path=ckpt_path,
        data_path='data/instruction/instructions.jsonl',
        tokenizer_path='data/tokenizer/mewtwo.model',
        output_dir='checkpoints/sft',
    )
    print('SFT complete!')
else:
    print('ERROR: No pretrained checkpoint found. Run Step 3 first.')

## Step 5: DPO Alignment

In [ ]:
#@title DPO on preference data { display-mode: "form" }
import json, os, sys, torch

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.alignment.dpo import train_dpo

# Create preference dataset
os.makedirs('data/preference', exist_ok=True)
preferences = [
    {'prompt': 'What is ML?', 'chosen': 'Machine learning is a subset of AI where systems learn patterns from data.', 'rejected': 'idk lol its when computers learn stuff'},
    {'prompt': 'Explain gradient descent.', 'chosen': 'Gradient descent iteratively adjusts parameters to minimize loss.', 'rejected': 'gradient descent is when you go downhill on a graph'},
    {'prompt': 'What is attention?', 'chosen': 'Attention computes weighted sums of input tokens using query-key-value pairs.', 'rejected': 'attention is when the model pays attention to words'},
]
with open('data/preference/preferences.jsonl', 'w') as f:
    for item in preferences:
        f.write(json.dumps(item) + '\n')

config = MewtwoConfig()

# Find SFT checkpoint
ckpt_path = None
for name in ['mewtwo_sft.pt', 'mewtwo_final.pt', 'model.pt']:
    p = f'checkpoints/sft/{name}'
    if os.path.exists(p):
        ckpt_path = p
        break

if ckpt_path:
    train_dpo(
        config=config,
        sft_model_path=ckpt_path,
        data_path='data/preference/preferences.jsonl',
        tokenizer_path='data/tokenizer/mewtwo.model',
        output_dir='checkpoints/dpo',
    )
    print('DPO complete!')
else:
    print('ERROR: No SFT checkpoint found. Run Step 4 first.')

## Step 6: Generate text

In [ ]:
#@title Generate with trained model { display-mode: "form" }
import torch, sys, os

sys.path.insert(0, '.')
from config.model_config import MewtwoConfig
from src.model.pocketllm import MewtwoLLM
from src.tokenizer.tokenizer import load_tokenizer
from src.inference.generate import generate

prompt = "What is the meaning of life?"  #@param {type:"string"}
max_tokens = 200  #@param {type:"integer"}
temperature = 0.8  #@param {type:"number"}

config = MewtwoConfig()
tokenizer = load_tokenizer('data/tokenizer/mewtwo.model')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MewtwoLLM(config).to(device)

# Load best checkpoint
for stage in ['dpo', 'sft', 'pretrain']:
    for name in ['mewtwo_final.pt', 'mewtwo_best.pt', 'model.pt']:
        ckpt = f'checkpoints/{stage}/{name}'
        if os.path.exists(ckpt):
            checkpoint = torch.load(ckpt, map_location=device, weights_only=False)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            print(f'Loaded: {ckpt}')
            break
    else:
        continue
    break

model.eval()
print(f'\nPrompt: {prompt}')
print('=' * 60)

output = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=max_tokens,
    temperature=temperature,
    top_k=50,
    top_p=0.95,
)
print(output)
print('=' * 60)